# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện Python; PyTorch CUDA đã được cài vào kernel local.
%pip -q install timm openpyxl pandas numpy

import sys, os, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Đường dẫn local trên máy này.
REPO_DIR = Path(r"C:\Users\Ta Quang Dung\CODE\VinAI\LabWork\K4-Track4-Day2-Deeplearning-Advance")
SUBMISSION_DIR = REPO_DIR / "submissions" / "2A202602588_TaQuangDung"
CODE_DIR = SUBMISSION_DIR / "code"
if not REPO_DIR.is_dir():
    raise FileNotFoundError(f"Repo không tồn tại: {REPO_DIR}")
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f"Thư mục code không tồn tại: {CODE_DIR}")
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))  # để import eval.py
sys.path.insert(0, str(CODE_DIR))  # để import code bài làm

print("Repo:", REPO_DIR)
print("Code:", CODE_DIR)
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib
import urllib.request
import zipfile

DATA_DIR = REPO_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
labels_dir = DATA_DIR / "labels"
labels_dir.mkdir(parents=True, exist_ok=True)

# Download archive if needed, always verify MD5, and extract only if images are missing.
images_dir = DATA_DIR / "images"
image_files = list(images_dir.rglob("*.jpg")) if images_dir.is_dir() else []
zip_path = DATA_DIR / "images.zip"
temp_path = DATA_DIR / "images.zip.part"
if not zip_path.exists():
    url = "https://zenodo.org/records/7939060/files/images.zip?download=1"
    print("Đang tải images.zip (~490 MB)...")
    urllib.request.urlretrieve(url, temp_path)
    temp_path.replace(zip_path)

md5 = hashlib.md5()
with zip_path.open("rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        md5.update(chunk)
actual_md5 = md5.hexdigest()
expected_md5 = "b7b30f96d466fba86016aa5a26606e0f"
print("images.zip MD5:", actual_md5)
assert actual_md5 == expected_md5, f"MD5 sai: {actual_md5}; hãy tải lại {zip_path}."

if len(image_files) != 17509:
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATA_DIR)

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    csv_path = labels_dir / f"{name}.csv"
    if not csv_path.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}.csv", csv_path)

image_files = list((DATA_DIR / "images").rglob("*.jpg"))
print("Số ảnh .jpg:", len(image_files))
print("Một vài ảnh:", *[str(path) for path in image_files[:5]], sep="\n")
print("CSV:", sorted(path.name for path in labels_dir.glob("*.csv")))

In [ ]:
IMAGES_DIR = str(REPO_DIR / "data" / "images")
LABELS_DIR = str(REPO_DIR / "data" / "labels")
assert len(list(Path(IMAGES_DIR).rglob("*.jpg"))) == 17509, "Kiểm tra lại IMAGES_DIR: cần thấy đủ 17.509 ảnh."
print("IMAGES_DIR:", IMAGES_DIR)
print("LABELS_DIR:", LABELS_DIR)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import dataset

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
labels_df = pd.read_csv(Path(LABELS_DIR) / "labels.csv")
class_names = labels_df.drop_duplicates("Label").sort_values("Label")["Species"].tolist()
assert len(class_names) == dataset.NUM_CLASSES
split_frames = {"train": train_df, "val": val_df, "test": test_df}
counts = pd.DataFrame({
    split: frame["Label"].value_counts().reindex(range(dataset.NUM_CLASSES), fill_value=0).to_numpy()
    for split, frame in split_frames.items()
}, index=class_names)
print("Số ảnh mỗi split:", split_report["n"])
print("Số ảnh theo lớp (số đếm thực):\n", counts)
print("Giao:", split_report["overlap"], "| Hợp:", split_report["union"], "| Thiếu:", split_report["missing"])

fig, ax = plt.subplots(figsize=(13, 5))
counts.plot(kind="bar", ax=ax)
ax.set(title="DeepWeeds fold 0 — class distribution", xlabel="Class", ylabel="Number of images")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
plt.show()

all_counts = labels_df["Label"].value_counts().reindex(range(dataset.NUM_CLASSES), fill_value=0)
print(f"Toàn bộ labels.csv: lớp lớn nhất/lớp nhỏ nhất = {all_counts.max() / all_counts.min():.2f} "
      f"({all_counts.max()}/{all_counts.min()}).")
print("Table 1 tham chiếu: Negatives 9,106; mỗi loài khoảng 1,009–1,125 ảnh.")

sample_groups = [train_df[train_df["Label"] == label].sample(3, random_state=0)
                 for label in range(dataset.NUM_CLASSES)]
fig, axes = plt.subplots(dataset.NUM_CLASSES, 3, figsize=(9, 24))
for label, group in enumerate(sample_groups):
    for column, (_, row) in enumerate(group.iterrows()):
        with Image.open(Path(IMAGES_DIR) / row["Filename"]) as image:
            axes[label, column].imshow(image.convert("RGB"))
        axes[label, column].axis("off")
        if column == 0:
            axes[label, column].set_ylabel(class_names[label], fontsize=9)
fig.suptitle("Three train images per class — inspect similarities and Negative examples")
fig.tight_layout()
plt.show()

image_shapes, image_modes = set(), set()
for image_path in Path(IMAGES_DIR).glob("*.jpg"):
    with Image.open(image_path) as image:
        image_shapes.add(image.size)
        image_modes.add(image.mode)
print("Kích thước ảnh (width, height):", sorted(image_shapes))
print("PIL modes / kênh trước chuyển RGB:", sorted(image_modes))
print("Dataset chuyển ảnh về RGB; chuẩn hóa ImageNet mean/std.")

In [ ]:
import torch
import dataset, losses, model as model_lib, train as train_lib

train_lib.set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
aug_dataset = dataset.DeepWeedsDataset(
    train_df, IMAGES_DIR, dataset.build_transforms(True, img_size=128, aug="basic")
)
preview_x, preview_y, _ = next(iter(dataset.make_loader(
    train_df.head(8), IMAGES_DIR, dataset.build_transforms(True, 128, "basic"),
    batch_size=6, train=False, num_workers=0,
)))
mean = torch.tensor(dataset.IMAGENET_MEAN).view(1, 3, 1, 1)
std = torch.tensor(dataset.IMAGENET_STD).view(1, 3, 1, 1)
preview = (preview_x * std + mean).clamp(0, 1)
fig, axes = plt.subplots(1, len(preview), figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(preview[i].permute(1, 2, 0))
    ax.set_title(class_names[int(preview_y[i])])
    ax.axis("off")
fig.suptitle("Training augmentation preview (denormalized)")
fig.tight_layout()
plt.show()

mini_df = train_df.groupby("Label", group_keys=False).head(1).head(8).reset_index(drop=True)
mini_ds = dataset.DeepWeedsDataset(
    mini_df, IMAGES_DIR, dataset.build_transforms(False, img_size=96)
)
mini_x = torch.stack([mini_ds[i][0] for i in range(len(mini_ds))]).to(device)
mini_y = torch.tensor(mini_df["Label"].to_numpy(), dtype=torch.long, device=device)
net = model_lib.build_model("resnet18", pretrained=False, num_classes=9, init="scratch").to(device)
ce = losses.build_criterion("ce")
net.eval()
with torch.inference_mode():
    initial_loss = float(ce(net(mini_x), mini_y))
print(f"Initial CE on 9 classes: {initial_loss:.4f}; uniform reference = {np.log(9):.4f}")
assert abs(initial_loss - float(np.log(9))) < 0.8, "Initial CE is unexpectedly far from ln(9)."

net.train()
optimizer = torch.optim.AdamW(net.parameters(), lr=0.01, weight_decay=0.0)
for step in range(30):
    optimizer.zero_grad(set_to_none=True)
    loss = ce(net(mini_x), mini_y)
    loss.backward()
    optimizer.step()
print(f"Small-batch overfit loss after 30 steps: {float(loss.detach()):.6f}")
assert float(loss.detach()) < 0.1, "Small batch did not overfit; investigate pipeline before experiments."

frozen_net = model_lib.build_model("resnet18", pretrained=False, num_classes=9, init="frozen").to(device)
frozen_opt = train_lib.build_optimizer(
    frozen_net, train_lib.Config(lr_backbone=1e-4, lr_head=1e-3)
)
frozen_loader = torch.utils.data.DataLoader(mini_ds, batch_size=2, shuffle=False)
frozen_cfg = train_lib.Config(epochs=1, batch_size=2, amp=False)
frozen_sched = train_lib.build_scheduler(frozen_opt, frozen_cfg, steps_per_epoch=1)
frozen_scaler = torch.amp.GradScaler("cuda", enabled=False)
train_lib.train_one_epoch(
    frozen_net, frozen_loader, ce, frozen_opt, frozen_sched, frozen_scaler,
    frozen_cfg, device,
)
bn_training = [module.training for module in frozen_net.modules()
               if isinstance(module, torch.nn.modules.batchnorm._BatchNorm)]
assert not any(bn_training), "Frozen backbone BatchNorm unexpectedly entered train mode."
train_lib.evaluate(frozen_net, frozen_loader, ce, device)
assert not frozen_net.training
print("Pipeline checks passed: fixed seed, CE scale, small-batch overfit, eval/train and frozen BatchNorm modes.")

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
import subprocess
results_path = SUBMISSION_DIR / "results.xlsx"
if not results_path.is_file() or len(pd.read_excel(results_path, sheet_name="Backbones")) != 5:
    subprocess.run([sys.executable, str(CODE_DIR / "run_backbones.py")], cwd=REPO_DIR, check=True)
backbone_results = pd.read_excel(results_path, sheet_name="Backbones")
display(backbone_results.sort_values("val_macro_f1", ascending=False))

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
from train import Config, run
# Bước 2 — kiểm soát thí nghiệm theo nguyên tắc 1 thay đổi 1 yếu tố
# Dùng 1 backbone được chọn ở Bước 1; nếu có backbone_results thì lấy backbone tốt nhất.
# B??c 2 ? ki?m so?t th? nghi?m theo nguy?n t?c 1 thay ??i 1 y?u t?
# D?ng 1 backbone ???c ch?n ? B??c 1; n?u c? backbone_results th? l?y backbone t?t nh?t.
backbone_choice = "convnext_tiny"
if "backbone_results" in globals():
    backbone_choice = str(backbone_results.sort_values("val_macro_f1", ascending=False).iloc[0]["backbone"]).strip()

base_cfg = dict(
    exp_id="T00",
    seed=0,
    backbone=backbone_choice,
    init="finetune",
    drop_rate=0.0,
    img_size=224,
    aug="basic",
    sampler=None,
    mix=None,
    mix_alpha=1.0,
    loss="ce",
    label_smoothing=0.0,
    focal_gamma=2.0,
    class_weight_beta=None,
    epochs=12,
    batch_size=32,
    grad_accum_steps=1,
    lr_backbone=1e-4,
    lr_head=1e-3,
    weight_decay=0.05,
    warmup_epochs=1.0,
    ema_decay=None,
    amp=True,
    num_workers=2,
    images_dir=str(DATA_DIR / "images"),
    labels_dir=str(DATA_DIR / "labels"),
    out_dir=str(SUBMISSION_DIR / "runs"),
    pred_dir=str(SUBMISSION_DIR / "predictions"),
    save_test_predictions=False,
)

experiments = [
    {"exp_id": "T00", "axis": "baseline", "change_vs_T00": "baseline", "cfg": {**base_cfg, "exp_id": "T00"}},
    {"exp_id": "T01", "axis": "init", "change_vs_T00": "init=frozen (ch? train head)", "cfg": {**base_cfg, "exp_id": "T01", "init": "frozen"}},
    {"exp_id": "T02", "axis": "loss", "change_vs_T00": "loss=ls, label_smoothing=0.1", "cfg": {**base_cfg, "exp_id": "T02", "loss": "ls", "label_smoothing": 0.1}},
    {"exp_id": "T03", "axis": "loss", "change_vs_T00": "loss=focal, gamma=2.0", "cfg": {**base_cfg, "exp_id": "T03", "loss": "focal", "focal_gamma": 2.0}},
    {"exp_id": "T04", "axis": "aug", "change_vs_T00": "aug=color", "cfg": {**base_cfg, "exp_id": "T04", "aug": "color"}},
    {"exp_id": "T05", "axis": "mix", "change_vs_T00": "mix=cutmix", "cfg": {**base_cfg, "exp_id": "T05", "mix": "cutmix", "mix_alpha": 1.0}},
    {"exp_id": "T06", "axis": "combined", "change_vs_T00": "aug=color + cutmix + focal + EMA", "cfg": {**base_cfg, "exp_id": "T06", "aug": "color", "mix": "cutmix", "mix_alpha": 1.0, "loss": "focal", "focal_gamma": 2.0, "ema_decay": 0.999}},
]

baseline_f1 = None
rows = []
for item in experiments:
    cfg = Config(**item["cfg"])
    result = run(cfg)
    val_f1 = float(result["val_macro_f1"])
    if item["exp_id"] == "T00":
        baseline_f1 = val_f1
    delta = val_f1 - baseline_f1 if baseline_f1 is not None else 0.0
    rows.append({
        "exp_id": item["exp_id"],
        "axis": item["axis"],
        "change_vs_T00": item["change_vs_T00"],
        "seed": cfg.seed,
        "val_macro_f1": val_f1,
        "val_top1": float(result["val_top1"]),
        "delta_vs_T00": delta,
        "note": "baseline" if item["exp_id"] == "T00" else "one-factor ablation",
    })
    print(f"{item['exp_id']}: val_macro_f1={val_f1:.4f}, delta_vs_T00={delta:.4f}, backbone={cfg.backbone}")

training_table = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
display(training_table[["exp_id", "axis", "change_vs_T00", "val_macro_f1", "val_top1", "delta_vs_T00"]])

# Ghi v?o sheet Training n?u file ?? t?n t?i; n?u ch?a, t?o m?i.
results_path = SUBMISSION_DIR / "results.xlsx"
if results_path.exists():
    from openpyxl import load_workbook
    wb = load_workbook(results_path)
    if "Training" in wb.sheetnames:
        del wb["Training"]
    wb.save(results_path)
with pd.ExcelWriter(results_path, engine="openpyxl", mode="a" if results_path.exists() else "w") as writer:
    training_table.to_excel(writer, sheet_name="Training", index=False)

print(f"Saved Part 2 ablation rows to {results_path}")
print("K?t lu?n: ch? coi l? c?i thi?n khi ? so v?i T00 r? r?ng v? v??t ng??ng nhi?u (~0.2?0.3 macro-F1).")


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# B??c 3 ? Suy lu?n v? ?o ?? tr?
# M?c ti?u: d?ng l?i checkpoint ?? train, kh?ng train l?i. So s?nh c?c ph??ng ph?p suy lu?n + ?? tr? tr?n val.

import pandas as pd
import torch
import numpy as np
from pathlib import Path

import model as model_lib
import eval as eval_lib
from dataset import load_split, make_loader, build_transforms, NUM_CLASSES
from inference import predict_logits, view_hflip, aggregate_views, fit_temperature, apply_temperature
from benchmark import latency_report

# Ch?n m? h?nh t?t nh?t t? B??c 1/2; s?a n?u b?n d?ng exp_id kh?c.
# Chọn mô hình tốt nhất từ Bước 1/2 (mặc định B03 hoặc T04)
best_exp = "B03"
best_backbone = "convnext_tiny"
checkpoint_path = SUBMISSION_DIR / "outputs" / "runs" / best_exp / "seed0" / "best.pt"
if not checkpoint_path.is_file():
    checkpoint_path = SUBMISSION_DIR / "runs" / best_exp / "seed0" / "best.pt"

# Load val split v? loader.
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
val_loader = make_loader(
    val_df, IMAGES_DIR, build_transforms(False, 224, aug="basic"),
    batch_size=32, train=False, sampler=None, num_workers=2,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model = model_lib.build_model(
    best_backbone, pretrained=False, num_classes=NUM_CLASSES, drop_rate=0.0, init="finetune"
).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()

def eval_metrics_from_logits(logits, y_true):
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=1).numpy()
    return eval_lib.compute_metrics(y_true, probs.argmax(1), probs)

# I00: 1 view (m?c)
names, y_true, logits_base = predict_logits(model, val_loader, device, view=None)
probs_base = torch.softmax(torch.tensor(logits_base, dtype=torch.float32), dim=1).numpy()
metrics_I00 = eval_metrics_from_logits(logits_base, y_true)
print("I00:", metrics_I00)

# I01: TTA l?t ngang (K=2)
names_h, y_true_h, logits_hflip = predict_logits(model, val_loader, device, view=view_hflip)
probs_hflip = torch.softmax(torch.tensor(logits_hflip, dtype=torch.float32), dim=1).numpy()
probs_I01 = aggregate_views([logits_base, logits_hflip], space="logit")
metrics_I01 = eval_lib.compute_metrics(y_true, probs_I01.argmax(1), probs_I01)
print("I01:", metrics_I01)

# I03: g?p x?c su?t tr?n 2 view
probs_I03 = aggregate_views([probs_base, probs_hflip], space="prob")
metrics_I03 = eval_lib.compute_metrics(y_true, probs_I03.argmax(1), probs_I03)
print("I03:", metrics_I03)

# I04: ??i ?? ph?n gi?i l?n 256 ?? ki?m tra FixRes
val_loader_256 = make_loader(
    val_df, IMAGES_DIR, build_transforms(False, 256, aug="basic"),
    batch_size=32, train=False, sampler=None, num_workers=2,
)
names_256, y_true_256, logits_256 = predict_logits(model, val_loader_256, device, view=None)
metrics_I04 = eval_metrics_from_logits(logits_256, y_true_256)
print("I04_256:", metrics_I04)

# I07: temperature scaling tr?n val; T kh?ng ???c fit tr?n test
T = fit_temperature(logits_base, y_true)
probs_T = apply_temperature(logits_base, T)
metrics_I07 = eval_lib.compute_metrics(y_true, probs_T.argmax(1), probs_T)
print("T =", T)
print("I07:", metrics_I07)

# ?o ?? tr? batch 1 v? batch 32 theo quy t?c: warmup, synchronize, p50/p95/p99
lat_1 = latency_report(model, batch_size=1, img_size=224, dtype="fp32", device=device.type, warmup=10, iters=100)
lat_32 = latency_report(model, batch_size=32, img_size=224, dtype="fp32", device=device.type, warmup=10, iters=100)
print("lat_1:", lat_1)
print("lat_32:", lat_32)

# Ghi k?t qu? v?o file results.xlsx d??i sheet Inference v? Latency
inference_rows = [
    {"method": "I00", "macro_f1": float(metrics_I00["macro_f1"]), "top1": float(metrics_I00["top1"]), "note": "1 view baseline"},
    {"method": "I01", "macro_f1": float(metrics_I01["macro_f1"]), "top1": float(metrics_I01["top1"]), "note": "TTA flip"},
    {"method": "I03", "macro_f1": float(metrics_I03["macro_f1"]), "top1": float(metrics_I03["top1"]), "note": "avg probability"},
    {"method": "I04", "macro_f1": float(metrics_I04["macro_f1"]), "top1": float(metrics_I04["top1"]), "note": "resize 256"},
    {"method": "I07", "macro_f1": float(metrics_I07["macro_f1"]), "top1": float(metrics_I07["top1"]), "note": "temperature scaling"},
]

results_xlsx = SUBMISSION_DIR / "results.xlsx"
if results_xlsx.exists():
    from openpyxl import load_workbook
    wb = load_workbook(results_xlsx)
    for sheet_name in ("Inference", "Latency"):
        if sheet_name in wb.sheetnames:
            del wb[sheet_name]
    wb.save(results_xlsx)

with pd.ExcelWriter(results_xlsx, engine="openpyxl", mode="a" if results_xlsx.exists() else "w") as writer:
    pd.DataFrame(inference_rows).to_excel(writer, sheet_name="Inference", index=False)
    pd.DataFrame([lat_1, lat_32]).to_excel(writer, sheet_name="Latency", index=False)

print("Saved Inference and Latency sheets to", results_xlsx)
print("K?t lu?n: TTA/ensemble t?ng ?? ch?nh x?c nh?ng chi ph? g?n tuy?n t?nh; tr?n robot ?u ti?n EMA/FP16 ho?c kh?ng TTA.")


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
from train import Config, run
# Chốt cấu hình trên val trước khi mở test.
# -- Dùng cấu hình đã chọn ở Bước 1/2; đây là ví dụ chung, sửa nếu cần.
# Ch?t c?u h?nh tr?n val tr??c khi m? test.
# -- D?ng c?u h?nh ?? ch?n ? B??c 1/2; ??y l? v? d? chung, s?a n?u c?n.
final_exp = "F01"
final_backbone = best_backbone if "best_backbone" in globals() else "convnext_tiny"
final_cfg_template = dict(
    exp_id=final_exp,
    seed=0,
    backbone=final_backbone,
    init="finetune",
    drop_rate=0.0,
    img_size=224,
    aug="basic",
    sampler=None,
    mix=None,               # n?u c?u h?nh t?t nh?t c? CutMix/Mixup, thay b?ng "cutmix" ho?c "mixup"
    mix_alpha=1.0,
    loss="ce",
    label_smoothing=0.0,
    focal_gamma=2.0,
    class_weight_beta=None,
    epochs=12,
    batch_size=32,
    grad_accum_steps=1,
    lr_backbone=1e-4,
    lr_head=1e-3,
    weight_decay=0.05,
    warmup_epochs=1.0,
    ema_decay=None,
    amp=True,
    num_workers=2,
    images_dir=str(DATA_DIR / "images"),
    labels_dir=str(DATA_DIR / "labels"),
    out_dir=str(SUBMISSION_DIR / "runs"),
    pred_dir=str(SUBMISSION_DIR / "predictions"),
    save_test_predictions=True,
)

for seed in (0, 1, 2):
    run(Config(**{**final_cfg_template, "seed": seed, "exp_id": final_exp}))
    run(Config(
        exp_id="T00",
        seed=seed,
        backbone="resnet50",
        init="finetune",
        images_dir=str(DATA_DIR / "images"),
        labels_dir=str(DATA_DIR / "labels"),
        out_dir=str(SUBMISSION_DIR / "runs"),
        pred_dir=str(SUBMISSION_DIR / "predictions"),
        save_test_predictions=True,
    ))

print("?? ch?y chung k?t v? m?c ? 3 seed.")


In [ ]:
# T?nh ch? s? theo ??ng ??nh ngh?a c?a l?p (?? vi?t s?n, kh?ng s?a eval.py).
# D?ng subprocess ?? ???ng d?n Windows c? d?u c?ch v?n ch?y ??ng.
import subprocess
for tag in ("F01", "T00"):
    subprocess.run([
        sys.executable, str(REPO_DIR / "eval.py"), "score",
        "--pred", str(SUBMISSION_DIR / "predictions" / f"{tag}_seed*_test.csv"),
        "--test-csv", str(Path(LABELS_DIR) / "test_subset0.csv"),
        "--labels", str(Path(LABELS_DIR) / "labels.csv"),
        "--tag", tag, "--out", str(SUBMISSION_DIR / "eval_out"),
    ], check=True)


In [ ]:
# T? ch?m ph?n I c?a RUBRIC (?? xu?t, gi?ng vi?n x?c nh?n).
# N?u ?? c? val predictions, c? th? th?m --uncal v? --final-val ?? ch?m I4(a/b).
subprocess.run([
    sys.executable, str(REPO_DIR / "eval.py"), "grade",
    "--final", str(SUBMISSION_DIR / "predictions" / "F01_seed*_test.csv"),
    "--baseline", str(SUBMISSION_DIR / "predictions" / "T00_seed*_test.csv"),
    "--test-csv", str(Path(LABELS_DIR) / "test_subset0.csv"),
    "--labels", str(Path(LABELS_DIR) / "labels.csv"),
    "--out", str(SUBMISSION_DIR / "eval_out"),
], check=True)


## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# T?o workbook k?t qu?, ki?m tra curve/prediction v? vi?t khung b?o c?o.
# B??c n?y ch?y sau khi ?? c? d? li?u th?t t? B??c 1?4.
from pathlib import Path
import re
import pandas as pd
from openpyxl import load_workbook

results_xlsx = SUBMISSION_DIR / "results.xlsx"
curves_dir = SUBMISSION_DIR / "curves"
preds_dir = SUBMISSION_DIR / "predictions"
report_path = SUBMISSION_DIR / "report.md"

# Khung sheet theo GUIDE.md m?c 6.1
sheet_columns = {
    "Backbones": [
        "exp_id", "backbone", "pretrained_tag", "params_m", "gmacs", "img_size",
        "epochs", "seed", "val_macro_f1", "val_top1", "epoch_seconds_mean",
        "batch1_latency_ms", "note"
    ],
    "Training": [
        "exp_id", "backbone", "axis", "change_vs_T00", "seed", "val_macro_f1",
        "val_top1", "delta_vs_T00", "f1_rare_classes", "note"
    ],
    "Inference": [
        "exp_id", "method", "checkpoint", "K", "val_macro_f1", "val_top1",
        "val_ece", "p50_ms", "p95_ms", "p99_ms", "throughput_img_s", "relative_cost_vs_I00"
    ],
    "Final": [
        "exp_id", "config", "seed", "val_macro_f1", "test_macro_f1", "test_top1",
        "test_ece", "mean_std_over_seeds", "note"
    ],
    "PerClass": [
        "class_name", "n_test", "precision", "recall", "f1", "config", "split"
    ],
    "Latency": [
        "config", "gpu", "dtype", "batch_size", "fused_bn", "p50_ms", "p95_ms", "p99_ms",
        "throughput_img_s"
    ],
    "Summary": [
        "rank", "exp_id", "config", "val_macro_f1", "test_macro_f1", "p95_ms", "note"
    ],
}

frames = {name: pd.DataFrame(columns=cols) for name, cols in sheet_columns.items()}

if results_xlsx.exists():
    for sheet_name in frames:
        try:
            existing = pd.read_excel(results_xlsx, sheet_name=sheet_name)
            if not existing.empty:
                frames[sheet_name] = existing.reindex(columns=sheet_columns[sheet_name])
        except Exception:
            frames[sheet_name] = pd.DataFrame(columns=sheet_columns[sheet_name])

# Ki?m tra curves/ v? predictions theo exp_id
curve_paths = sorted(curves_dir.glob("*.png")) if curves_dir.exists() else []
curve_names = [p.name for p in curve_paths]
curve_exp_ids = sorted({re.sub(r"(?:_[^\/]+)?\.png$", "", p.name) for p in curve_paths})

# N?u c? summary ho?c k?t qu? t? c?c exp_id ?? l?u, gi? nguy?n; n?u ch?a c?, t?o khung b?o c?o r?ng.
report_lines = [
    "# Report",
    "",
    "## 1) T?m t?t",
    "- B?i to?n: DeepWeeds 9-class plant weed classification.",
    "- M?c ti?u: x?c ??nh backbone, c?ng th?c hu?n luy?n v? suy lu?n t?t nh?t tr?n val, sau ?? b?o c?o test m?t l?n.",
    "- C?u h?nh t?t nh?t: c?n ?i?n sau khi ch?y xong final experiment v? c? s? li?u th?t t? eval.py.",
    "",
    "## 2) D? li?u v? thi?t l?p",
    "- Fold: 0 (train_subset0.csv, val_subset0.csv, test_subset0.csv)",
    "- ?? ph?n b? l?p: c?n bi?u ?? EDA v? b?ng s? ??m th?t t? notebook.",
    "- Ph?n c?ng: {gpu_name}",
    "- Th? vi?n: PyTorch {torch_version}, timm {timm_version}, pandas {pd.__version__}",
    "",
    "## 3) K?t qu? backbone",
    "- B?ng Backbones trong results.xlsx; ch?n backbone d?a tr?n val_macro_f1 v? ?? tr? batch-1.",
    "",
    "## 4) K?t qu? c?ng th?c hu?n luy?n",
    "- B?ng Training trong results.xlsx; so s?nh delta v?i T00 v? x?t nhi?u.",
    "",
    "## 5) K?t qu? suy lu?n",
    "- B?ng Inference v? Latency; so s?nh macro-F1, ECE, p95 v? chi ph? t??ng ??i.",
    "",
    "## 6) C?u h?nh t?t nh?t",
    "- Final model: c?n ?i?n ch?nh x?c sau khi c? final seed v? eval.py score/grade.",
    "",
    "## 7) K?t lu?n",
    "- N?i th?n tr?ng: n?u ch?nh l?ch nh? h?n nhi?u th? ghi ?kh?ng ph?n bi?t ???c?.",
    "",
    "## 8) H?n ch?",
    "- Ch? c? 1 fold, 3 seed, v? d? li?u chia ng?u nhi?n; kh?ng kh?ng ??nh t?ng qu?t tr?n d? li?u m?i.",
    "",
    "## 9) Ph? l?c",
    "- exp_id ?? ch?y: " + (", ".join(curve_exp_ids) if curve_exp_ids else "ch?a c?") + ".",
    "- C?c file d? ?o?n: " + (", ".join(sorted(p.name for p in preds_dir.glob("*.csv"))) if preds_dir.exists() else "ch?a c?") + ".",
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")

# L?u workbook, lu?n ??m b?o sheet ??ng t?n v? c?t
with pd.ExcelWriter(results_xlsx, engine="openpyxl") as writer:
    for name, df in frames.items():
        df = df.copy()
        if list(df.columns) != sheet_columns[name]:
            for col in sheet_columns[name]:
                if col not in df.columns:
                    df[col] = None
            df = df.reindex(columns=sheet_columns[name])
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for column_cells in ws.columns:
            width = min(max(len(str(cell.value or "")) for cell in column_cells) + 2, 40)
            ws.column_dimensions[column_cells[0].column_letter].width = width

# Ki?m tra san b?ng s? li?u gi?a workbook v? file th?c t?
missing_curve = []
for exp in curve_exp_ids:
    if not any(exp in p.name for p in curve_paths):
        missing_curve.append(exp)

missing_pred = []
if preds_dir.exists():
    pred_files = sorted(preds_dir.glob("*.csv"))
    for p in pred_files:
        if p.name.endswith("_test.csv") and not p.exists():
            missing_pred.append(p.name)

print("Workbook saved:", results_xlsx)
print("Curve PNG count:", len(curve_paths))
print("Prediction CSV count:", len(sorted((preds_dir).glob("*.csv"))) if preds_dir.exists() else 0)
print("Curve exp ids:", curve_exp_ids)
print("Missing curve refs:", missing_curve)
print("Report written:", report_path)
print("N?u ?? ch?y xong final test v? eval.py score/grade, h?y ki?m tra l?i t?ng sheet ?? ??i chi?u s? li?u th?t.")
